Import 

In [80]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)



Import data

In [81]:

df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

df["date"] = pd.to_datetime(df["date"])

df = (
    df
    .sort_values("date")
    .reset_index(drop=True)
)

print("Shape:", df.shape)
display(df.head())

Shape: (1887, 4)


,date,pm1,pm2_5,pm10
0,2016-08-25,144.026083,187.599837,269.346300
1,2016-08-27,121.881565,159.537520,230.975440
2,2016-08-28,98.196718,124.696523,171.362190
3,2016-08-29,54.083770,63.915300,78.042140
4,2016-09-07,127.947061,168.501950,245.585955


In [82]:
df = df[["date", "pm2_5", "pm10"]].copy()

display(df.head())

,date,pm2_5,pm10
0,2016-08-25,187.599837,269.346300
1,2016-08-27,159.537520,230.975440
2,2016-08-28,124.696523,171.362190
3,2016-08-29,63.915300,78.042140
4,2016-09-07,168.501950,245.585955


Create 7 days lags

In [83]:
for lag in range(1, 8):
    df[f"pm2_5_lag_{lag}"] = df["pm2_5"].shift(lag)
    df[f"pm10_lag_{lag}"] = df["pm10"].shift(lag)

Create forecasting target

In [84]:
df["target_pm2_5"] = df["pm2_5"].shift(-1)

Remove unusable rows 

In [85]:
df_model = df.dropna().copy()

print("Original rows:", len(df))
print("Usable rows:", len(df_model))

display(df_model.head())

Original rows: 1887
Usable rows: 1879


,date,pm2_5,pm10,pm2_5_lag_1,pm10_lag_1,pm2_5_lag_2,pm10_lag_2,pm2_5_lag_3,pm10_lag_3,pm2_5_lag_4,pm10_lag_4,pm2_5_lag_5,pm10_lag_5,pm2_5_lag_6,pm10_lag_6,pm2_5_lag_7,pm10_lag_7,target_pm2_5
7,2016-09-12,132.785217,182.608250,29.042917,32.999425,63.253007,83.520250,168.501950,245.585955,63.915300,78.042140,124.696523,171.362190,159.537520,230.975440,187.599837,269.346300,119.215280
8,2016-09-18,119.215280,166.891925,132.785217,182.608250,29.042917,32.999425,63.253007,83.520250,168.501950,245.585955,63.915300,78.042140,124.696523,171.362190,159.537520,230.975440,96.302953
9,2016-09-19,96.302953,130.395945,119.215280,166.891925,132.785217,182.608250,29.042917,32.999425,63.253007,83.520250,168.501950,245.585955,63.915300,78.042140,124.696523,171.362190,104.368733
10,2016-09-20,104.368733,147.140180,96.302953,130.395945,119.215280,166.891925,132.785217,182.608250,29.042917,32.999425,63.253007,83.520250,168.501950,245.585955,63.915300,78.042140,106.495640
11,2016-09-21,106.495640,145.790225,104.368733,147.140180,96.302953,130.395945,119.215280,166.891925,132.785217,182.608250,29.042917,32.999425,63.253007,83.520250,168.501950,245.585955,103.276103


## Rolling Stats

Import and sorting 

In [86]:
rolling_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

rolling_df["date"] = pd.to_datetime(
    rolling_df["date"]
)

rolling_df = (
    rolling_df[
        ["date", "pm2_5", "pm10"]
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

display(rolling_df.head())

,date,pm2_5,pm10
0,2016-08-25,187.599837,269.346300
1,2016-08-27,159.537520,230.975440
2,2016-08-28,124.696523,171.362190
3,2016-08-29,63.915300,78.042140
4,2016-09-07,168.501950,245.585955


Create target 

In [87]:
rolling_df["target_pm2_5"] = (rolling_df["pm2_5"].shift(-1))

Create rolling features with 3 day and 7 day  mean and std

In [88]:
for pollutant in ["pm2_5", "pm10"]:

    rolling_df[f"{pollutant}_rolling_mean_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .mean()
    )

    rolling_df[f"{pollutant}_rolling_mean_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .mean()
    )

    rolling_df[f"{pollutant}_rolling_std_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .std()
    )

    rolling_df[f"{pollutant}_rolling_std_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .std()
    )

adding min and max too 

In [89]:
for pollutant in ["pm2_5", "pm10"]:

    rolling_df[f"{pollutant}_rolling_min_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .min()
    )

    rolling_df[f"{pollutant}_rolling_min_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .min()
    )

    rolling_df[f"{pollutant}_rolling_max_3"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=3)
        .max()
    )

    rolling_df[f"{pollutant}_rolling_max_7"] = (
        rolling_df[pollutant]
        .shift(1)
        .rolling(window=7)
        .max()
    )

In [90]:
print("Shape:", rolling_df.shape)

display(
    rolling_df[
        [
            "date",
            "pm2_5",
            "pm10",
            "pm2_5_rolling_mean_3",
            "pm2_5_rolling_mean_7",
            "pm10_rolling_mean_3",
            "pm10_rolling_mean_7",
            "target_pm2_5",
        ]
    ].head(10)
)

Shape: (1887, 20)


,date,pm2_5,pm10,pm2_5_rolling_mean_3,pm2_5_rolling_mean_7,pm10_rolling_mean_3,pm10_rolling_mean_7,target_pm2_5
0,2016-08-25,187.599837,269.346300,NaN,NaN,NaN,NaN,159.537520
1,2016-08-27,159.537520,230.975440,NaN,NaN,NaN,NaN,124.696523
2,2016-08-28,124.696523,171.362190,NaN,NaN,NaN,NaN,63.915300
3,2016-08-29,63.915300,78.042140,157.277960,NaN,223.894643,NaN,168.501950
4,2016-09-07,168.501950,245.585955,116.049781,NaN,160.126590,NaN,63.253007
5,2016-09-10,63.253007,83.520250,119.037924,NaN,164.996762,NaN,29.042917
6,2016-09-11,29.042917,32.999425,98.556752,NaN,135.716115,NaN,132.785217
7,2016-09-12,132.785217,182.608250,86.932624,113.792436,120.701877,158.833100,119.215280
8,2016-09-18,119.215280,166.891925,75.027047,105.961776,99.709308,146.441950,96.302953
9,2016-09-19,96.302953,130.395945,93.681138,100.201456,127.499867,137.287162,104.368733


Remove unusable rows 

In [91]:
rolling_model_df = (
    rolling_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(rolling_df))
print("Usable rows:", len(rolling_model_df))

Original rows: 1887
Usable rows: 1879


EWMA (Exponentially Weighted Moving Average)  

Import data and sorting

In [92]:
ewma_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

ewma_df["date"] = pd.to_datetime(
    ewma_df["date"]
)

ewma_df = (
    ewma_df[
        ["date", "pm2_5", "pm10"]
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

Create target 

In [93]:
ewma_df["target_pm2_5"] = (
    ewma_df["pm2_5"].shift(-1)
)

Create features 

In [94]:
for pollutant in ["pm2_5", "pm10"]:

    ewma_df[f"{pollutant}_ewma_3"] = (
        ewma_df[pollutant]
        .shift(1)
        .ewm(span=3, adjust=False)
        .mean()
    )

    ewma_df[f"{pollutant}_ewma_7"] = (
        ewma_df[pollutant]
        .shift(1)
        .ewm(span=7, adjust=False)
        .mean()
    )

In [95]:
display(
    ewma_df[
        [
            "date",
            "pm2_5",
            "pm10",
            "pm2_5_ewma_3",
            "pm2_5_ewma_7",
            "pm10_ewma_3",
            "pm10_ewma_7",
            "target_pm2_5",
        ]
    ].head(10)
)

,date,pm2_5,pm10,pm2_5_ewma_3,pm2_5_ewma_7,pm10_ewma_3,pm10_ewma_7,target_pm2_5
0,2016-08-25,187.599837,269.346300,NaN,NaN,NaN,NaN,159.537520
1,2016-08-27,159.537520,230.975440,187.599837,187.599837,269.346300,269.346300,124.696523
2,2016-08-28,124.696523,171.362190,173.568678,180.584258,250.160870,259.753585,63.915300
3,2016-08-29,63.915300,78.042140,149.132601,166.612324,210.761530,237.655736,168.501950
4,2016-09-07,168.501950,245.585955,106.523950,140.938068,144.401835,197.752337,63.253007
5,2016-09-10,63.253007,83.520250,137.512950,147.829038,194.993895,209.710742,29.042917
6,2016-09-11,29.042917,32.999425,100.382978,126.685031,139.257072,178.163119,132.785217
7,2016-09-12,132.785217,182.608250,64.712948,102.274502,86.128249,141.872195,119.215280
8,2016-09-18,119.215280,166.891925,98.749082,109.902181,134.368249,152.056209,96.302953
9,2016-09-19,96.302953,130.395945,108.982181,112.230456,150.630087,155.765138,104.368733


Remove unusable rows 

In [96]:
ewma_model_df = (
    ewma_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(ewma_df))
print("Usable rows:", len(ewma_model_df))

Original rows: 1887
Usable rows: 1885


Change/ Trend features 

Load data and sorting

In [97]:
change_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

change_df["date"] = pd.to_datetime(
    change_df["date"]
)

change_df = (
    change_df[
        ["date", "pm2_5", "pm10"]
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

Create tomorrow's target

In [98]:
change_df["target_pm2_5"] = (
    change_df["pm2_5"].shift(-1)
)

Create change features

In [99]:
for pollutant in ["pm2_5", "pm10"]:

    change_df[f"{pollutant}_change_1"] = (
        change_df[pollutant]
        - change_df[pollutant].shift(1)
    )

    change_df[f"{pollutant}_change_3"] = (
        change_df[pollutant]
        - change_df[pollutant].shift(3)
    )

    change_df[f"{pollutant}_change_7"] = (
        change_df[pollutant]
        - change_df[pollutant].shift(7)
    )

Calculate change percentage too

In [100]:
for pollutant in ["pm2_5", "pm10"]:

    change_df[f"{pollutant}_pct_change_1"] = (
        change_df[pollutant]
        .pct_change(1)
    )

    change_df[f"{pollutant}_pct_change_3"] = (
        change_df[pollutant]
        .pct_change(3)
    )

    change_df[f"{pollutant}_pct_change_7"] = (
        change_df[pollutant]
        .pct_change(7)
    )

Check features

In [101]:
display(
    change_df[
        [
            "date",
            "pm2_5",
            "pm10",
            "pm2_5_change_1",
            "pm2_5_change_3",
            "pm2_5_change_7",
            "pm10_change_1",
            "pm10_change_3",
            "pm10_change_7",
            "target_pm2_5",
        ]
    ].head(10)
)

,date,pm2_5,pm10,pm2_5_change_1,pm2_5_change_3,pm2_5_change_7,pm10_change_1,pm10_change_3,pm10_change_7,target_pm2_5
0,2016-08-25,187.599837,269.346300,NaN,NaN,NaN,NaN,NaN,NaN,159.537520
1,2016-08-27,159.537520,230.975440,-28.062317,NaN,NaN,-38.370860,NaN,NaN,124.696523
2,2016-08-28,124.696523,171.362190,-34.840997,NaN,NaN,-59.613250,NaN,NaN,63.915300
3,2016-08-29,63.915300,78.042140,-60.781223,-123.684537,NaN,-93.320050,-191.304160,NaN,168.501950
4,2016-09-07,168.501950,245.585955,104.586650,8.964430,NaN,167.543815,14.610515,NaN,63.253007
5,2016-09-10,63.253007,83.520250,-105.248943,-61.443517,NaN,-162.065705,-87.841940,NaN,29.042917
6,2016-09-11,29.042917,32.999425,-34.210090,-34.872383,NaN,-50.520825,-45.042715,NaN,132.785217
7,2016-09-12,132.785217,182.608250,103.742300,-35.716733,-54.81462,149.608825,-62.977705,-86.738050,119.215280
8,2016-09-18,119.215280,166.891925,-13.569937,55.962273,-40.32224,-15.716325,83.371675,-64.083515,96.302953
9,2016-09-19,96.302953,130.395945,-22.912327,67.260037,-28.39357,-36.495980,97.396520,-40.966245,104.368733


Remove unusable rows

In [102]:
change_model_df = (
    change_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(change_df))
print("Usable rows:", len(change_model_df))

Original rows: 1887
Usable rows: 1879


Combined one : 7-day lags, rolling mean/std, EWMA, change/trend

In [103]:
combined_df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

combined_df["date"] = pd.to_datetime(
    combined_df["date"]
)

combined_df = (
    combined_df[
        ["date", "pm2_5", "pm10"]
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

print("Shape:", combined_df.shape)

display(combined_df.head())

Shape: (1887, 3)


,date,pm2_5,pm10
0,2016-08-25,187.599837,269.346300
1,2016-08-27,159.537520,230.975440
2,2016-08-28,124.696523,171.362190
3,2016-08-29,63.915300,78.042140
4,2016-09-07,168.501950,245.585955


Create target

In [104]:
combined_df["target_pm2_5"] = (
    combined_df["pm2_5"].shift(-1)
)

add 7 days lag

In [105]:
for lag in range(1, 8):

    combined_df[f"pm2_5_lag_{lag}"] = (
        combined_df["pm2_5"].shift(lag)
    )

    combined_df[f"pm10_lag_{lag}"] = (
        combined_df["pm10"].shift(lag)
    )

add rolling features

In [106]:
for pollutant in ["pm2_5", "pm10"]:

    combined_df[f"{pollutant}_rolling_mean_3"] = (
        combined_df[pollutant]
        .shift(1)
        .rolling(3)
        .mean()
    )

    combined_df[f"{pollutant}_rolling_mean_7"] = (
        combined_df[pollutant]
        .shift(1)
        .rolling(7)
        .mean()
    )

    combined_df[f"{pollutant}_rolling_std_7"] = (
        combined_df[pollutant]
        .shift(1)
        .rolling(7)
        .std()
    )

add ewma

In [107]:
for pollutant in ["pm2_5", "pm10"]:

    combined_df[f"{pollutant}_ewma_3"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=3, adjust=False)
        .mean()
    )

    combined_df[f"{pollutant}_ewma_7"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=7, adjust=False)
        .mean()
    )

add change and trends

In [108]:
for pollutant in ["pm2_5", "pm10"]:

    combined_df[f"{pollutant}_ewma_3"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=3, adjust=False)
        .mean()
    )

    combined_df[f"{pollutant}_ewma_7"] = (
        combined_df[pollutant]
        .shift(1)
        .ewm(span=7, adjust=False)
        .mean()
    )

combined feature count

In [109]:
feature_columns = [
    col
    for col in combined_df.columns
    if col not in [
        "date",
        "pm2_5",
        "pm10",
        "target_pm2_5",
    ]
]

print("Total features:", len(feature_columns))

print("\nFeatures:")
for feature in feature_columns:
    print(feature)

Total features: 24

Features:
pm2_5_lag_1
pm10_lag_1
pm2_5_lag_2
pm10_lag_2
pm2_5_lag_3
pm10_lag_3
pm2_5_lag_4
pm10_lag_4
pm2_5_lag_5
pm10_lag_5
pm2_5_lag_6
pm10_lag_6
pm2_5_lag_7
pm10_lag_7
pm2_5_rolling_mean_3
pm2_5_rolling_mean_7
pm2_5_rolling_std_7
pm10_rolling_mean_3
pm10_rolling_mean_7
pm10_rolling_std_7
pm2_5_ewma_3
pm2_5_ewma_7
pm10_ewma_3
pm10_ewma_7


remove unusable rows 

In [110]:
combined_model_df = (
    combined_df
    .dropna()
    .reset_index(drop=True)
)

print("Original rows:", len(combined_df))
print("Usable rows:", len(combined_model_df))
print("Features:", len(feature_columns))

Original rows: 1887
Usable rows: 1879
Features: 24


In [111]:
print("Missing values:")
print(
    combined_model_df[
        feature_columns + ["target_pm2_5"]
    ]
    .isna()
    .sum()
    .sum()
)

print("\nFinal shape:")
print(combined_model_df.shape)

display(combined_model_df.head())

Missing values:
0

Final shape:
(1879, 28)


,date,pm2_5,pm10,target_pm2_5,pm2_5_lag_1,pm10_lag_1,pm2_5_lag_2,pm10_lag_2,pm2_5_lag_3,pm10_lag_3,...,pm2_5_rolling_mean_3,pm2_5_rolling_mean_7,pm2_5_rolling_std_7,pm10_rolling_mean_3,pm10_rolling_mean_7,pm10_rolling_std_7,pm2_5_ewma_3,pm2_5_ewma_7,pm10_ewma_3,pm10_ewma_7
0,2016-09-12,132.785217,182.608250,119.215280,29.042917,32.999425,63.253007,83.520250,168.501950,245.585955,...,86.932624,113.792436,61.747630,120.701877,158.833100,94.104586,64.712948,102.274502,86.128249,141.872195
1,2016-09-18,119.215280,166.891925,96.302953,132.785217,182.608250,29.042917,32.999425,63.253007,83.520250,...,75.027047,105.961776,53.790609,99.709308,146.441950,82.068412,98.749082,109.902181,134.368249,152.056209
2,2016-09-19,96.302953,130.395945,104.368733,119.215280,166.891925,132.785217,182.608250,29.042917,32.999425,...,93.681138,100.201456,49.046922,127.499867,137.287162,74.270861,108.982181,112.230456,150.630087,155.765138
3,2016-09-20,104.368733,147.140180,106.495640,96.302953,130.395945,119.215280,166.891925,132.785217,182.608250,...,116.101150,96.145232,47.842838,159.965373,131.434841,72.736509,102.642567,108.248580,140.513016,149.422840
4,2016-09-21,106.495640,145.790225,103.276103,104.368733,147.140180,96.302953,130.395945,119.215280,166.891925,...,106.628989,101.924294,45.695910,148.142683,141.305990,68.868714,103.505650,107.278618,143.826598,148.852175


datasets into dictionaries

In [112]:
feature_datasets = {
    "Lag 7": df_model,
    "Rolling": rolling_model_df,
    "EWMA": ewma_model_df,
    "Change": change_model_df,
    "Combined": combined_model_df,
}

determine features

In [113]:
feature_columns_map = {}

for name, data in feature_datasets.items():

    features = [
        col
        for col in data.columns
        if col not in [
            "date",
            "pm2_5",
            "pm10",
            "target_pm2_5",
        ]
    ]

    feature_columns_map[name] = features

    print(
        f"{name}: {len(features)} features"
    )

Lag 7: 14 features
Rolling: 16 features
EWMA: 4 features
Change: 12 features
Combined: 24 features


Chronological 80/20 split

In [114]:
splits = {}

for name, data in feature_datasets.items():

    split_index = int(len(data) * 0.8)

    train_df = data.iloc[:split_index].copy()
    val_df = data.iloc[split_index:].copy()
    features = feature_columns_map[name]
    X_train = train_df[features]
    y_train = train_df["target_pm2_5"]

    X_val = val_df[features]
    y_val = val_df["target_pm2_5"]

    splits[name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
    }

    print(
        f"{name}: "
        f"Train={len(train_df)}, "
        f"Validation={len(val_df)}"
    )

Lag 7: Train=1503, Validation=376
Rolling: Train=1503, Validation=376
EWMA: Train=1508, Validation=377
Change: Train=1503, Validation=376
Combined: Train=1503, Validation=376


Define our models

In [115]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(alpha=1.0),

    "Random Forest 100": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    ),

    "Random Forest 300": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),

    "Random Forest 500": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    ),

    "Gradient Boosting 100": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),

    "Gradient Boosting 200": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),

    "Gradient Boosting 300": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),

    "Extra Trees 100": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees 300": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees 500": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    ),
}

Train everything 

In [116]:
results = []

trained_models = {}

for feature_name, split in splits.items():

    X_train = split["X_train"]
    y_train = split["y_train"]

    X_val = split["X_val"]
    y_val = split["y_val"]

    trained_models[feature_name] = {}

    print("\n" + "=" * 70)
    print(f"FEATURE SET: {feature_name}")
    print("=" * 70)

    for model_name, model in models.items():

        print(f"Training: {model_name}")

        model.fit(
            X_train,
            y_train,
        )

        predictions = model.predict(X_val)

        mae = mean_absolute_error(
            y_val,
            predictions,
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                predictions,
            )
        )

        r2 = r2_score(
            y_val,
            predictions,
        )

        results.append({
            "Feature Set": feature_name,
            "Model": model_name,
            "Features": len(
                feature_columns_map[feature_name]
            ),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

        trained_models[
            feature_name
        ][model_name] = model


FEATURE SET: Lag 7
Training: Linear Regression
Training: Ridge
Training: Random Forest 100
Training: Random Forest 300
Training: Random Forest 500
Training: Gradient Boosting 100
Training: Gradient Boosting 200
Training: Gradient Boosting 300
Training: Extra Trees 100
Training: Extra Trees 300
Training: Extra Trees 500

FEATURE SET: Rolling
Training: Linear Regression
Training: Ridge
Training: Random Forest 100
Training: Random Forest 300
Training: Random Forest 500
Training: Gradient Boosting 100
Training: Gradient Boosting 200
Training: Gradient Boosting 300
Training: Extra Trees 100
Training: Extra Trees 300
Training: Extra Trees 500

FEATURE SET: EWMA
Training: Linear Regression
Training: Ridge
Training: Random Forest 100
Training: Random Forest 300
Training: Random Forest 500
Training: Gradient Boosting 100
Training: Gradient Boosting 200
Training: Gradient Boosting 300
Training: Extra Trees 100
Training: Extra Trees 300
Training: Extra Trees 500

FEATURE SET: Change
Training: Li

comparison table

In [117]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "RMSE"
).reset_index(drop=True)

display(results_df)

,Feature Set,Model,Features,MAE,RMSE,R2
0,EWMA,Linear Regression,4,10.700978,14.376865,0.789240
1,EWMA,Ridge,4,10.701023,14.376887,0.789240
2,Lag 7,Ridge,14,10.776943,14.473274,0.786725
3,Lag 7,Linear Regression,14,10.776975,14.473345,0.786723
4,Combined,Ridge,24,10.866083,14.856402,0.775284
5,Combined,Linear Regression,24,10.924535,14.916927,0.773449
6,Rolling,Ridge,16,12.695255,17.638498,0.683241
7,Rolling,Linear Regression,16,12.697634,17.645239,0.682998
8,Lag 7,Gradient Boosting 100,14,14.176324,18.410990,0.654888
9,Change,Random Forest 300,12,15.123276,18.647929,0.645948


Rounding the errors 

In [118]:
results_display = results_df.copy()

results_display["MAE"] = (
    results_display["MAE"].round(3)
)

results_display["RMSE"] = (
    results_display["RMSE"].round(3)
)

results_display["R2"] = (
    results_display["R2"].round(3)
)

display(results_display)

,Feature Set,Model,Features,MAE,RMSE,R2
0,EWMA,Linear Regression,4,10.701,14.377,0.789
1,EWMA,Ridge,4,10.701,14.377,0.789
2,Lag 7,Ridge,14,10.777,14.473,0.787
3,Lag 7,Linear Regression,14,10.777,14.473,0.787
4,Combined,Ridge,24,10.866,14.856,0.775
5,Combined,Linear Regression,24,10.925,14.917,0.773
6,Rolling,Ridge,16,12.695,17.638,0.683
7,Rolling,Linear Regression,16,12.698,17.645,0.683
8,Lag 7,Gradient Boosting 100,14,14.176,18.411,0.655
9,Change,Random Forest 300,12,15.123,18.648,0.646


best model for each feature strategy 

In [119]:
best_by_feature = (
    results_df
    .sort_values("RMSE")
    .groupby("Feature Set")
    .first()
    .reset_index()
)

display(
    best_by_feature[
        [
            "Feature Set",
            "Model",
            "Features",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
)

,Feature Set,Model,Features,MAE,RMSE,R2
0,Change,Random Forest 300,12,15.123276,18.647929,0.645948
1,Combined,Ridge,24,10.866083,14.856402,0.775284
2,EWMA,Linear Regression,4,10.700978,14.376865,0.789240
3,Lag 7,Ridge,14,10.776943,14.473274,0.786725
4,Rolling,Ridge,16,12.695255,17.638498,0.683241


Find the overall winner

In [120]:
best_result = (
    results_df
    .sort_values("RMSE")
    .iloc[0]
)

print("BEST MODEL without pm1")
print("-" * 40)

print("Feature Set:", best_result["Feature Set"])
print("Model:", best_result["Model"])
print("Features:", best_result["Features"])
print("MAE:", round(best_result["MAE"], 3))
print("RMSE:", round(best_result["RMSE"], 3))
print("R²:", round(best_result["R2"], 3))

BEST MODEL without pm1
----------------------------------------
Feature Set: EWMA
Model: Linear Regression
Features: 4
MAE: 10.701
RMSE: 14.377
R²: 0.789
